# Prepare Qwen2.5 0.5B Instruct as a saved TensorRT-LLM LLM API engine

This notebook downloads the Hugging Face model, selects the TensorRT engine implementation of the LLM API, saves it with `LLM.save()`, and completes a native Triton TensorRT-LLM backend repository.

Prepare requires one GPU. Use the same image tag and compatible GPU architecture for build and deploy.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tempfile

EXAMPLE_DIR_NAME = "qwen2.5-0.5b-instruct-llmapi-engine-s3"
PROJECT = Path.cwd().resolve()
if PROJECT.name != EXAMPLE_DIR_NAME:
    raise RuntimeError(f"Run this notebook from {EXAMPLE_DIR_NAME}, got {PROJECT}")
if PROJECT.parent.name == EXAMPLE_DIR_NAME:
    raise RuntimeError(f"Nested example folder is wrong: {PROJECT}. Use one {EXAMPLE_DIR_NAME} folder only.")

HF_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
HF_MODEL_DIR = PROJECT / "hf_model"
MODEL_NAME = "qwen2_5_0_5b_instruct_llmapi_engine"
TRITON_MODEL_DIR = PROJECT / MODEL_NAME
VERSION_DIR = TRITON_MODEL_DIR / "1"
ENGINE_DIR = VERSION_DIR
LEGACY_MODEL_DIR = PROJECT / "tensorrt_llm"

# Older notebook revisions created this folder. Its name shadows the installed
# tensorrt_llm Python package when notebook runs from PROJECT.
if LEGACY_MODEL_DIR.exists():
    shutil.rmtree(LEGACY_MODEL_DIR)
    print("Removed stale legacy model directory:", LEGACY_MODEL_DIR)

for stale_work in Path(tempfile.gettempdir()).glob("qwen_llmapi_engine_notebook_*"):
    shutil.rmtree(stale_work, ignore_errors=True)
NOTEBOOK_WORK = Path(tempfile.mkdtemp(prefix="qwen_llmapi_engine_notebook_"))
LOCAL_HOME = NOTEBOOK_WORK / "home"
LOCAL_CACHE = NOTEBOOK_WORK / "cache"
HF_CACHE = LOCAL_CACHE / "huggingface"
PIP_CACHE = LOCAL_CACHE / "pip"
for path in (LOCAL_HOME, LOCAL_CACHE, HF_CACHE, PIP_CACHE):
    path.mkdir(parents=True, exist_ok=True)

os.environ["USER"] = "workspace"
os.environ["LOGNAME"] = "workspace"
os.environ["HOME"] = str(LOCAL_HOME)
os.environ["XDG_CACHE_HOME"] = str(LOCAL_CACHE)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTHONNOUSERSITE"] = "1"

# Jupyter may run from system Python although TensorRT-LLM is installed in
# the container's Triton virtual environment. Add its matching site-packages.
triton_site_packages = (
    Path("/opt/venv-tritonserver/lib")
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
)
if triton_site_packages.exists() and str(triton_site_packages) not in sys.path:
    sys.path.insert(0, str(triton_site_packages))
    print("Added Triton virtual environment packages:", triton_site_packages)

print("PROJECT:", PROJECT)
print("HF_MODEL_DIR:", HF_MODEL_DIR)
print("TRITON_MODEL_DIR:", TRITON_MODEL_DIR)
print("ENGINE_DIR:", ENGINE_DIR)


In [ ]:
# Validate the checked-in native TensorRT-LLM repository skeleton.
for required in (TRITON_MODEL_DIR / "config.pbtxt",):
    if not required.exists():
        raise RuntimeError(f"Missing checked-in Triton model file: {required}")
print("Using model repository skeleton:", TRITON_MODEL_DIR)


In [ ]:
# Use dependency versions bundled with TensorRT-LLM. Installing current
# Hugging Face packages here can override TensorRT-LLM's version constraints.
import importlib

stale_dependency_paths = [
    path for path in sys.path
    if "qwen_llmapi_engine_notebook_" in path and Path(path).name == "deps"
]
for path in stale_dependency_paths:
    sys.path.remove(path)
for module_name, module in list(sys.modules.items()):
    module_file = str(getattr(module, "__file__", "") or "")
    module_root = module_name.split(".", 1)[0]
    if (
        "qwen_llmapi_engine_notebook_" in module_file
        or module_root in {"huggingface_hub", "transformers", "requests", "certifi", "urllib3", "idna", "charset_normalizer"}
    ):
        del sys.modules[module_name]
for variable in ("REQUESTS_CA_BUNDLE", "CURL_CA_BUNDLE", "SSL_CERT_FILE"):
    if "qwen_llmapi_engine_notebook_" in os.environ.get(variable, ""):
        del os.environ[variable]
importlib.invalidate_caches()

import certifi
import huggingface_hub
print("Using bundled huggingface_hub:", huggingface_hub.__version__)
print("Using CA bundle:", certifi.where())
if not Path(certifi.where()).is_file():
    raise RuntimeError(f"CA bundle does not exist: {certifi.where()}")


In [ ]:
# Download Hugging Face model once. This folder is build input, not deployed runtime input.
from huggingface_hub import snapshot_download

HF_MODEL_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=HF_MODEL_ID,
    local_dir=HF_MODEL_DIR,
    local_dir_use_symlinks=False,
)

print("Downloaded files:")
for path in sorted(HF_MODEL_DIR.iterdir()):
    print(path.name)


In [ ]:
# Build and save TensorRT-LLM engine through LLM API. This cell needs a GPU.
if ENGINE_DIR.exists():
    shutil.rmtree(ENGINE_DIR)
ENGINE_DIR.mkdir(parents=True, exist_ok=True)
# Run LLM API in a clean Triton process, matching deployed Python models.
BUILD_SCRIPT = NOTEBOOK_WORK / "build_llmapi_engine.py"
BUILD_SCRIPT.write_text("""import os
import sys

from tensorrt_llm.llmapi.llm import _TrtLLM as LLM

def main():
    if len(sys.argv) != 3:
        raise RuntimeError(f"Expected model and engine paths, got: {sys.argv}")
    model_dir, engine_dir = sys.argv[1:3]
    llm = None
    try:
        llm = LLM(model=model_dir, tensor_parallel_size=1)
        if not hasattr(llm, "save"):
            raise RuntimeError("This LLM API implementation does not provide LLM.save()")
        llm.save(engine_dir)
    finally:
        if llm is not None and hasattr(llm, "shutdown"):
            llm.shutdown()

if __name__ == "__main__":
    main()
""", encoding="utf-8")
TRITON_PYTHON = Path("/opt/venv-tritonserver/bin/python")
if not TRITON_PYTHON.exists():
    raise RuntimeError(f"Triton Python executable missing: {TRITON_PYTHON}")
BUILD_DEPS = NOTEBOOK_WORK / "build_deps"
subprocess.check_call([
    str(TRITON_PYTHON), "-m", "pip", "install",
    "--disable-pip-version-check", "--no-deps",
    "--target", str(BUILD_DEPS), "openai==2.44.0",
])
build_env = os.environ.copy()
existing_pythonpath = build_env.get("PYTHONPATH", "")
build_env["PYTHONPATH"] = (
    str(BUILD_DEPS) if not existing_pythonpath
    else f"{BUILD_DEPS}:{existing_pythonpath}"
)
BUILD_LOG = PROJECT / "llmapi_engine_build.log"
with BUILD_LOG.open("w", encoding="utf-8") as log_file:
    build = subprocess.run(
        [str(TRITON_PYTHON), str(BUILD_SCRIPT), str(HF_MODEL_DIR), str(ENGINE_DIR)],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env=build_env,
        check=False,
    )
if build.returncode != 0:
    log_lines = BUILD_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    log_tail = "\n".join(log_lines[-120:])
    raise RuntimeError(
        f"LLM API engine build failed with exit code {build.returncode}. "
        f"Full log: {BUILD_LOG}\n\n{log_tail}"
    )
print("LLM API build log:", BUILD_LOG)

print("Saved engine files:")
for path in sorted(ENGINE_DIR.rglob("*")):
    print(path.relative_to(PROJECT))
if not any(ENGINE_DIR.iterdir()):
    raise RuntimeError(f"LLM.save() did not write files into {ENGINE_DIR}")


In [ ]:
# LLM.save() stores tokenizer files beside the engine. Remove duplicate folder
# created by older notebook revisions and verify saved tokenizer artifacts.
duplicate_tokenizer_dir = VERSION_DIR / "tokenizer"
shutil.rmtree(duplicate_tokenizer_dir, ignore_errors=True)

tokenizer_names = [
    "tokenizer.json",
    "tokenizer.model",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "generation_config.json",
    "vocab.json",
    "merges.txt",
]
print("Tokenizer files:")
saved_tokenizer_files = [VERSION_DIR / name for name in tokenizer_names if (VERSION_DIR / name).exists()]
if not saved_tokenizer_files:
    raise RuntimeError(f"LLM.save() did not store tokenizer files under {VERSION_DIR}")
for path in saved_tokenizer_files:
    print(path.relative_to(PROJECT))


In [ ]:
# Final repository check.
required = [
    TRITON_MODEL_DIR / "config.pbtxt",
    VERSION_DIR / "config.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError(f"Missing repository artifacts: {missing}")
engine_files = sorted(VERSION_DIR.glob("rank*.engine"))
if not engine_files:
    raise RuntimeError(f"No rank*.engine files found directly under {VERSION_DIR}")

print("Deploy repository root:", PROJECT)
print("Generated model repository:")
for path in sorted(TRITON_MODEL_DIR.rglob("*")):
    print(path.relative_to(PROJECT))

shutil.rmtree(HF_MODEL_DIR, ignore_errors=True)
shutil.rmtree(NOTEBOOK_WORK, ignore_errors=True)
BUILD_LOG.unlink(missing_ok=True)
print("Removed build-only Hugging Face model:", HF_MODEL_DIR)
print("Removed temporary notebook workspace:", NOTEBOOK_WORK)
print("Serving repository ready:", TRITON_MODEL_DIR)
